# Tutorial 09 — Comparative LCA & Visualization

Companion explainer: **09_comparative_lca_visualization.md**. Consolidate the
calculation patterns into a small comparative toolkit, and build the report
figures: grouped bars, a normalized heatmap, and a break-even line plot.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import bw2data as bd
import bw2calc as bc

bd.projects.set_current("bw25-tutorials")
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)

def find_flow(name, categories=("air",)):
    return next(f for f in bio if f["name"] == name and f["categories"] == categories)

co2 = find_flow("Carbon dioxide, fossil")
so2 = find_flow("Sulfur dioxide")
nox = find_flow("Nitrogen oxides")

## Three cup alternatives (functional unit: 1000 servings)
ceramic (durable, washed), paper (single-use), polystyrene (single-use).

In [2]:
DB = "t09_cups"
if DB in bd.databases:
    del bd.databases[DB]
bd.Database(DB).write({
    (DB, "elec"): {"name": "electricity", "unit": "kilowatt hour", "exchanges": [
        {"input": (DB, "elec"), "amount": 1.0, "type": "production"},
        {"input": co2.key, "amount": 0.5, "type": "biosphere"},
        {"input": so2.key, "amount": 0.0015, "type": "biosphere"},
        {"input": nox.key, "amount": 0.0012, "type": "biosphere"}]},
    (DB, "ceramic"): {"name": "ceramic mug (1000 uses)", "unit": "unit", "exchanges": [
        {"input": (DB, "ceramic"), "amount": 1.0, "type": "production"},
        {"input": (DB, "elec"), "amount": 3.5, "type": "technosphere"},  # mfg
        {"input": co2.key, "amount": 0.6, "type": "biosphere"},
        # washing: 1000 washes * 0.02 kWh
        {"input": (DB, "elec"), "amount": 20.0, "type": "technosphere"}]},
    (DB, "paper"): {"name": "paper cups (1000)", "unit": "unit", "exchanges": [
        {"input": (DB, "paper"), "amount": 1.0, "type": "production"},
        {"input": (DB, "elec"), "amount": 8.0, "type": "technosphere"},
        {"input": co2.key, "amount": 11.0, "type": "biosphere"},
        {"input": nox.key, "amount": 0.02, "type": "biosphere"}]},
    (DB, "ps"): {"name": "polystyrene cups (1000)", "unit": "unit", "exchanges": [
        {"input": (DB, "ps"), "amount": 1.0, "type": "production"},
        {"input": (DB, "elec"), "amount": 6.0, "type": "technosphere"},
        {"input": co2.key, "amount": 14.0, "type": "biosphere"},
        {"input": so2.key, "amount": 0.03, "type": "biosphere"}]},
})
alts = {"ceramic": bd.get_node(database=DB, code="ceramic"),
        "paper":   bd.get_node(database=DB, code="paper"),
        "polystyrene": bd.get_node(database=DB, code="ps")}

13:18:34-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

13:18:34-0400

 [

info     

] 

Vacuuming database            

## Comparative results cube -> tidy DataFrame (factorize once, switch methods)

In [3]:
def find_methods(*subs, exclude=("no LT",)):
    out = []
    for m in bd.methods:
        s = str(m).lower()
        if all(x.lower() in s for x in subs) and not any(e.lower() in s for e in exclude):
            out.append(m)
    return out

gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))
methods = [gwp]
for kw in [("acidification",), ("eutrophication", "freshwater"), ("particulate",)]:
    h = find_methods("recipe", "midpoint", *kw)
    if h:
        methods.append(h[0])

labels = list(alts); acts = [alts[k] for k in labels]
lca = bc.LCA({acts[0]: 1}, method=methods[0]); lca.lci(factorize=True); lca.lcia()
rows = []
for m in methods:
    lca.switch_method(m)
    unit = bd.Method(m).metadata.get("unit", "")
    short = m[-1] if len(m) > 1 else str(m)
    for lbl, act in zip(labels, acts):
        lca.lcia(demand={act.id: 1})
        rows.append({"alternative": lbl, "method": short, "unit": unit, "score": lca.score})
tidy = pd.DataFrame(rows)
wide = tidy.pivot(index="alternative", columns="method", values="score")
print(wide.round(3).to_string())

D:\01code\Projects\SDAI- Ecosystem\Brightway2\.venv\Lib\site-packages\bw2calc\lca.py:250: UserWarning: All values in characterization matrix are zero
  warnings.warn("All values in characterization matrix are zero")


method       freshwater eutrophication potential (FEP)  global warming potential (GWP100)  particulate matter formation potential (PMFP)  terrestrial acidification potential (TAP)
alternative                                                                                                                                                                        
ceramic                                            0.0                              12.35                                          0.013                                      0.045
paper                                              0.0                              15.00                                          0.007                                      0.023
polystyrene                                        0.0                              17.00                                          0.012                                      0.042

## Figure 1 — grouped bars per category

In [4]:
cats = tidy["method"].unique()
fig, axes = plt.subplots(1, len(cats), figsize=(3.2*len(cats), 3.4))
if len(cats) == 1:
    axes = [axes]
for ax, cat in zip(axes, cats):
    sub = tidy[tidy["method"] == cat]
    ax.bar(sub["alternative"], sub["score"], color=["#55A868", "#C44E52", "#8172B3"])
    ax.set_title(cat[:22], fontsize=9)
    ax.set_ylabel(sub["unit"].iloc[0][:18], fontsize=8)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
plt.tight_layout()
plt.savefig("tutorials_outputs_09_bars.png", dpi=120, bbox_inches="tight")
print("saved tutorials_outputs_09_bars.png")
plt.show()

saved tutorials_outputs_09_bars.png

C:\Users\derne\AppData\Local\Temp\ipykernel_61580\2009887915.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 2 — normalized heatmap (each category scaled to its max)

In [5]:
norm = wide.div(wide.max(axis=0), axis=1)
fig, ax = plt.subplots(figsize=(6, 3))
im = ax.imshow(norm.values, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(norm.columns)))
ax.set_xticklabels([c[:14] for c in norm.columns], rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(len(norm.index))); ax.set_yticklabels(norm.index)
for i in range(norm.shape[0]):
    for j in range(norm.shape[1]):
        ax.text(j, i, f"{norm.values[i,j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Relative impact (1.0 = worst per category)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig("tutorials_outputs_09_heatmap.png", dpi=120, bbox_inches="tight")
print("saved tutorials_outputs_09_heatmap.png")
plt.show()

saved tutorials_outputs_09_heatmap.png

C:\Users\derne\AppData\Local\Temp\ipykernel_61580\2336512604.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 3 — break-even: ceramic GWP amortized over N uses vs single-use
The ceramic FU above is 1000 uses; show per-use GWP as a function of lifetime.

In [6]:
ceramic_total = wide.loc["ceramic", gwp[-1]]        # for 1000 uses
paper_total = wide.loc["paper", gwp[-1]]            # per 1000 cups
ps_total = wide.loc["polystyrene", gwp[-1]]
N = np.arange(1, 1001)
ceramic_per_use = ceramic_total / N * (1000 / 1000)  # mfg+wash spread over N uses (approx)
# Simplify: manufacturing fixed, washing per use. Approximate per-use curves:
paper_per_use = np.full_like(N, paper_total / 1000, dtype=float)
ps_per_use = np.full_like(N, ps_total / 1000, dtype=float)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(N, ceramic_per_use, label="ceramic (amortized)", color="#55A868")
ax.plot(N, paper_per_use, "--", label="paper (per use)", color="#C44E52")
ax.plot(N, ps_per_use, "--", label="polystyrene (per use)", color="#8172B3")
ax.set_xlabel("number of uses of the ceramic mug")
ax.set_ylabel("GWP per use (kg CO2-eq)")
ax.set_title("Break-even: when does the durable option win?")
ax.set_ylim(0, max(paper_per_use[0], ps_per_use[0]) * 1.5)
ax.legend()
plt.tight_layout()
plt.savefig("tutorials_outputs_09_breakeven.png", dpi=120, bbox_inches="tight")
print("saved tutorials_outputs_09_breakeven.png")
plt.show()

saved tutorials_outputs_09_breakeven.png

C:\Users\derne\AppData\Local\Temp\ipykernel_61580\3980879613.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export

In [7]:
wide.to_csv("tutorials_outputs_09_results.csv")
print("saved tutorials_outputs_09_results.csv")
print("\nReminder: always report functional unit (1000 servings) + method units.")

saved tutorials_outputs_09_results.csv


Reminder: always report functional unit (1000 servings) + method units.

Next: **10 — advanced internals**.